# **ASSOCIATION RULES (phân tích giỏ hàng)**

## **1. Tải lại dataset đã được làm sạch**

In [1]:
import pandas as pd

# Tải lại file
df_clean = pd.read_csv('df_clean.csv')
df_clean.head()

,Unnamed: 0,transaction_id,transaction_date,transaction_time,transaction_qty,store_id,store_location,product_id,unit_price,product_category,product_type,product_detail,Revenue,Hour,Day,Month,Year
0,0,1,2023-01-01,1900-01-01 07:06:11,2,5,Lower Manhattan,32,3.0,Coffee,Gourmet brewed coffee,Ethiopia Rg,6.0,7,Sunday,January,2023
1,1,2,2023-01-01,1900-01-01 07:08:56,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg,6.2,7,Sunday,January,2023
2,2,3,2023-01-01,1900-01-01 07:14:04,2,5,Lower Manhattan,59,4.5,Drinking Chocolate,Hot chocolate,Dark chocolate Lg,9.0,7,Sunday,January,2023
3,3,4,2023-01-01,1900-01-01 07:20:24,1,5,Lower Manhattan,22,2.0,Coffee,Drip coffee,Our Old Time Diner Blend Sm,2.0,7,Sunday,January,2023
4,4,5,2023-01-01,1900-01-01 07:22:41,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg,6.2,7,Sunday,January,2023


## **2. Chạy thuật toán Apriori để tìm các cặp quy luật kết hợp**

In [6]:
from mlxtend.frequent_patterns import apriori, association_rules

# ==========================================
# 0. TẠO MÃ HÓA ĐƠN GỘP (Receipt_ID)
# Nếu cùng Cửa hàng + Ngày + Giờ => Cùng 1 hóa đơn
# ==========================================
df_clean['Receipt_ID'] = (df_clean['store_id'].astype(str) + "_" +
                          df_clean['transaction_date'].astype(str) + "_" +
                          df_clean['transaction_time'].astype(str))

# 1. TẠO GIỎ HÀNG
basket = (df_clean.groupby(['Receipt_ID', 'product_detail'])['transaction_qty']
          .sum()
          .unstack()
          .reset_index()
          .fillna(0)
          .set_index('Receipt_ID'))

basket_sets = (basket > 0)

# 2. CHẠY THUẬT TOÁN APRIORI
# Do có hơn 140000 dòng nên đặt min_support xuống thấp 0.001 để nhận ra quy luật các cặp
frequent_itemsets = apriori(basket_sets, min_support=0.001, use_colnames=True)
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)

# 3. Xuất file csv kết quả
if rules.empty:
    print("Vẫn chưa tìm thấy quy luật nào!")
else:
    rules['Sản phẩm chính (Mua)'] = rules['antecedents'].apply(lambda x: ', '.join(list(x)))
    rules['Sản phẩm mua kèm'] = rules['consequents'].apply(lambda x: ', '.join(list(x)))

    # Trích xuất các chỉ số
    rules['Support (%)'] = round(rules['support'] * 100, 3)
    rules['Confidence (%)'] = round(rules['confidence'] * 100, 2)
    rules['Lift'] = round(rules['lift'], 2)

    # Lọc cột và sắp xếp ưu tiên hiển thị các quy luật có Lift mạnh nhất
    rules_final = rules[['Sản phẩm chính (Mua)', 'Sản phẩm mua kèm', 'Support (%)', 'Confidence (%)', 'Lift']]
    rules_final = rules_final.sort_values(by='Lift', ascending=False).reset_index(drop=True)

    # Hiển thị Top 15 quy luật để xem trước
    print(rules_final.head(15).to_string())

    # Xuất file CSV
    rules_final.to_csv('Cross_Selling_Rules.csv', index=False, encoding='utf-8-sig')
    print(f"=> Đã xuất thành công file 'Cross_Selling_Rules.csv' với {len(rules_final)} quy luật!")

        Sản phẩm chính (Mua)          Sản phẩm mua kèm  Support (%)  Confidence (%)   Lift
0       Ouro Brasileiro shot              Ginger Scone        0.589           31.29  15.82
1               Ginger Scone      Ouro Brasileiro shot        0.589           29.78  15.82
2                   Latte Rg            Hazelnut syrup        0.318           12.85   9.87
3             Hazelnut syrup                  Latte Rg        0.318           24.39   9.87
4              Cappuccino Lg           Chocolate syrup        0.324           13.68   9.22
5            Chocolate syrup             Cappuccino Lg        0.324           21.81   9.22
6                 Cappuccino  Sugar Free Vanilla syrup        0.339           14.30   9.22
7   Sugar Free Vanilla syrup                Cappuccino        0.339           21.88   9.22
8            Chocolate syrup                  Latte Rg        0.333           22.45   9.08
9                   Latte Rg           Chocolate syrup        0.333           13.47   9.08

# **PHÂN TÍCH INSIGHT VÀ GỢI Ý THÊM CHO SẢN PHẨM**

**1. Cặp sản phẩm thường xuyên được mua chung nhất**
* **Nhận xét:** Đáng chú ý nhất là cặp món **Ouro Brasileiro shot** (cafe) và **Ginger Scone** (bánh). Chỉ số Lift của cặp này lên tới 15.82, nghĩa là khi khách gọi món này thì xác suất họ mua món kia tăng lên gần 16 lần so với bình thường. Tỷ lệ Confidence cũng ở mức khá cao (~30%).
* **Đề xuất:**
  * Quán có thể tạo một "Combo Bữa Sáng" gồm 2 món này và giảm giá nhẹ để khuyến khích khách hàng mua cả hai.
  * Xếp bánh Ginger Scone ở vị trí dễ nhìn thấy ngay quầy thu ngân (gần máy pha cafe) để khách dễ dàng gọi thêm khi đang chờ lấy Ouro Brasileiro.

**2. Thói quen mua thêm hương liệu (Syrup) khi uống Cafe**
* **Nhận xét:** Từ luật số 2 đến số 14, có thể thấy khách hàng có xu hướng thích cho thêm Syrup vào các dòng cafe sữa. Các cặp này có chỉ số Lift khá đồng đều, dao động từ 8.7 đến 9.8.
    * Khách uống **Latte** thường hay gọi kèm **Hazelnut syrup** hoặc **Chocolate syrup**.
    * Khách uống **Cappuccino** rất hay gọi thêm **Sugar Free Vanilla syrup**, **Chocolate syrup** hoặc **Carmel syrup**.
* **Đề xuất:**
    * Nhắc nhở nhân viên thu ngân chủ động gợi ý cho khách. Ví dụ khi khách gọi Latte, nhân viên có thể hỏi thêm: *"Mình có muốn thêm chút Hazelnut hoặc Chocolate syrup cho đậm vị không ạ?"*. Do thói quen này đã có sẵn, tỷ lệ khách đồng ý sẽ rất cao.
    * Trên bảng Menu, ở ngay dưới tên món Latte và Cappuccino, có thể in thêm dòng chữ nhỏ gợi ý các loại Syrup phù hợp nhất để khách hàng dễ lựa chọn.